# Final pipeline: the numbers and figures of the report (CPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. **Run it once with `PILOT = True` first** (6 holdout files): it checks the plumbing, and its numbers are not a result.

It scores the fixed final pipeline on the date holdout with the quantities of the final evaluation (70 % quantitative, 30 % qualitative): PQ and mean Dice, the distributions of IoU and Dice, the one-to-many and many-to-one relations, and overlay figures of the mask morphology. The definitions are those of the organizers' self-evaluation notebook (<https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook>), implemented in `filament.final_metrics`; its module docstring says which function of that notebook each rule comes from and what the notebook does not define.

**The pipeline** (`docs/PROJECT_REPORT.md`): U-Net r34 80 epochs, flip TTA, threshold 0.45, close_size 11, min_area 400, solar-disk gate, object filter mean >= 0.8. The U-Net side is the probability cache of the Kaggle notebook `export-probs-r34`; no model runs here. The notebook recomputes the pooled PQ from the cache and stops if it differs from 0.4255 by more than 0.002 (not in a pilot).

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code`, a version that contains `filament.final_metrics`, `filament.final_eval` and `filament.final_figures`.
3. The output of the Kaggle notebook `ignacioaguirrecandia/export-probs-r34` (its cache for `MODEL = "r34"`).

**Settings:** Accelerator **None** (no GPU), Internet **Off** is enough. **Output** in `/kaggle/working`: `holdout_final_metrics.csv` (numbers only, so it can go in `results/`) and PNG figures: `iou_distribution.png`, `dice_distribution.png`, `relations.png` and one `overlay_<kind>.png` for each of good match, fragmentation, over-merge, miss and false positive. **The figures show frames of the dataset: do not publish them before the Rules and Data tabs have been read** (`AGENTS.md`, [VERIFICAR]); none is committed.

It reads only the holdout side of the cache. It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

## 1. `src` on the path

`src` is the directory that contains `filament/final_figures.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match.

In [ ]:
import os
import sys
from pathlib import Path

MARKER = Path("filament") / "final_figures.py"


def _local_src() -> Path | None:
    """A checkout that already has the module this notebook needs (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / MARKER).is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    The marker may sit at any depth. Directories named ``competitions`` are not
    walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / MARKER).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the private
    dataset's ``src`` is used in place. It is not copied to ``/kaggle/working``: a
    copy there would be saved with this notebook's output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            f"Could not find src/{MARKER.as_posix()} in the working directory or "
            "under /kaggle/input (any depth outside competitions). Attach the "
            "dataset ignacioaguirrecandia/solar-code. If it is attached, its "
            "version predates the final-evaluation modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings and the cache

- The pipeline settings are fixed in `filament.final_eval` (`FINAL_POINT` is `filament.gap_analysis.CURRENT_POINT`, the object filter is `ADOPTED_OBJECT_FILTER`). They are not settings of this notebook.
- The cache has to come from exactly the checkpoint `MODEL` names: its manifest goes through the same guard as a weights file. Leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.
- `EXAMPLES_PER_KIND` and `SEED` only choose which examples are drawn. The rule is in `filament.final_eval.select_examples`.

In [ ]:
# ---- Settings. Nothing below this cell needs editing. -----------------------
PILOT = False  # True: PILOT_FILES holdout files, to check the plumbing; not a result
PILOT_FILES = 6
MODEL = "r34"
ALLOW_UNVERIFIED_WEIGHTS = False
EXPECTED_BASE_PQ = 0.4255  # pooled PQ in sample of the final pipeline (docs/RESULTS.md)
BASE_PQ_TOLERANCE = 0.002
EXAMPLES_PER_KIND = 3  # overlay panels per kind
SEED = 0  # picks the overlay examples

In [ ]:
import time

from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    annotator_views,
    load_train_annotations,
)
from filament.baseline import on_disk
from filament.checkpoint import crop_checkpoint, verify_and_report
from filament.crop_data import read_gray
from filament.detector_data import build_detector_splits
from filament.detector_train import subsample_evenly
from filament.final_eval import (
    EXAMPLE_KINDS,
    FINAL_POINT,
    final_labels,
    frame_results,
    select_examples,
)
from filament.final_figures import (
    ATTRIBUTION,
    distribution_figure,
    overlay_figure,
    relations_figure,
    save_figure,
)
from filament.final_metrics import (
    format_summary,
    pooled_counts,
    pooled_pq,
    positive_scores,
    relation_counts,
    summary_rows,
    write_summary,
)
from filament.probcache import (
    ProbCache,
    find_cache_root,
    manifest_fingerprint,
    read_manifest,
    split_dir,
)
from filament.sweep import view_masks

EXPECTED = crop_checkpoint(MODEL)
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "final_evaluation"
)
OUTPUT_PATH = OUTPUT_DIR / "holdout_final_metrics.csv"

CACHE_ROOT = find_cache_root(MODEL, INPUT_ROOT)
print(f"cache={CACHE_ROOT}")
verify_and_report(
    manifest_fingerprint(read_manifest(CACHE_ROOT, MODEL)),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)
print(
    f"pipeline: threshold={FINAL_POINT.threshold} close_size={FINAL_POINT.close_size} "
    f"min_area={FINAL_POINT.min_area}"
)

## 3. Date holdout

The same split as the U-Net notebooks and the detector: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Every holdout file has to be in the cache.

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = build_detector_splits(loaded).holdout
views_total = sum(len(annotator_views(image)) for image in holdout)
print(f"holdout files={len(holdout)} annotator views={views_total}")
if not PILOT and (len(holdout), views_total) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views. [VERIFY] the data version."
    )
if PILOT:
    holdout = list(subsample_evenly(holdout, PILOT_FILES))
    print(f"PILOT: {len(holdout)} files; these numbers are not a result")
cache = ProbCache(split_dir(CACHE_ROOT, MODEL, "holdout"))
missing = [image.file_name for image in holdout if image.file_name not in cache]
if missing:
    raise RuntimeError(
        f"{len(missing)} holdout files are not in the cache, for example {missing[0]}."
    )

## 4. Score every annotator view

For each frame: the pipeline's labels from the cached probability, then the IoU and Dice of every (annotator mask, predicted instance) pair of every view (`filament.final_metrics.view_result`). PQ is pooled over the views as in the rest of the repo, and has to reproduce the documented number before anything else is written.

In [ ]:
results = []
begin = time.monotonic()
for index, image in enumerate(holdout, start=1):
    gray = read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))
    probability = cache.mean(image.file_name)
    if probability.shape != gray.shape:
        raise ValueError(
            f"{image.file_name}: probability {probability.shape} vs frame {gray.shape}"
        )
    labels = final_labels(gray, probability, on_disk(gray))
    results.extend(frame_results(image, labels))
    if index == 1 or index % 10 == 0 or index == len(holdout):
        print(
            f"  {index}/{len(holdout)} {image.file_name} ({time.monotonic() - begin:.0f} s)",
            flush=True,
        )

base_pq = pooled_pq(results)
totals = pooled_counts(results)
print(
    f"pooled PQ in sample: {base_pq:.4f} (expected {EXPECTED_BASE_PQ}); "
    f"TP={totals.tp} FP={totals.fp} FN={totals.fn}"
)
if not PILOT and not abs(base_pq - EXPECTED_BASE_PQ) <= BASE_PQ_TOLERANCE:
    raise RuntimeError(
        f"The pooled PQ {base_pq:.4f} differs from {EXPECTED_BASE_PQ} by more than "
        f"{BASE_PQ_TOLERANCE}: the cache, the data or the pipeline is not the documented one."
    )

## 5. The summary table

`source` says where each number comes from: `official` is computed or plotted by the organizers' notebook, `derived` follows from official counts (SQ and RQ are the Kirillov split of PQ, which that notebook does not report), `ours` is not in it (the quantiles, and `mean_dice_tp`, the mean Dice of the matched pairs only). `mean_dice` is the mean line of the official Dice histogram: the mean over every pair of an annotator mask and a prediction that touch, matched or not.

In [ ]:
rows = summary_rows(results)
write_summary(OUTPUT_PATH, rows)
print(format_summary(rows))
print(f"wrote {OUTPUT_PATH}")

## 6. Figures

The histograms and the relation chart follow `plot_distribution` and `plot_m2n_counts` of the official notebook. The overlay examples are chosen by `select_examples` (a fixed rule on the match degrees and the seed `SEED`, see its docstring), then drawn from the same pipeline output.

In [ ]:
iou_scores = positive_scores(results, "iou")
dice_scores = positive_scores(results, "dice")
relations = relation_counts(results)
written = [
    save_figure(
        distribution_figure(iou_scores, "IoU distribution (holdout)", "IoU"),
        OUTPUT_DIR / "iou_distribution.png",
    ),
    save_figure(
        distribution_figure(dice_scores, "Dice distribution (holdout)", "Dice"),
        OUTPUT_DIR / "dice_distribution.png",
    ),
    save_figure(
        relations_figure(relations, "Many to one and one to many (holdout)"),
        OUTPUT_DIR / "relations.png",
    ),
]

examples = select_examples(results, per_kind=EXAMPLES_PER_KIND, seed=SEED)
images_by_name = {image.file_name: image for image in holdout}
frames = {}  # file name -> (gray, labels); only the frames of the chosen examples
for example in examples:
    if example.file_name not in frames:
        image = images_by_name[example.file_name]
        gray = read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))
        labels = final_labels(gray, cache.mean(image.file_name), on_disk(gray))
        frames[example.file_name] = (gray, labels)
for kind in EXAMPLE_KINDS:
    panels = []
    for example in (e for e in examples if e.kind == kind):
        image = images_by_name[example.file_name]
        gray, labels = frames[example.file_name]
        views = annotator_views(image)
        panels.append(
            (example, gray, labels, view_masks(image, views[example.view_index]))
        )
    if not panels:
        print(f"no {kind} example in this holdout")
        continue
    written.append(
        save_figure(overlay_figure(kind, panels), OUTPUT_DIR / f"overlay_{kind}.png")
    )
    del panels
for path in written:
    print(f"wrote {path} ({path.stat().st_size // 1024} KiB)")

**Figure captions** (each PNG also carries the attribution in its corner). Replace the numbers by those of the printed summary when the figures go into the report.

- `iou_distribution.png`, `dice_distribution.png`: IoU and Dice of every pair of an annotator mask and a predicted filament that overlap, on the date holdout; the dashed line is the mean. Pairs that do not touch are left out, as in the organizers' notebook. GONG data, NSO/AURA/NSF.
- `relations.png`: number of predictions each annotator filament overlaps (right, `1:n`) and number of annotator filaments each prediction overlaps (left, `n:1`); `1:0` and `0:1` are misses and false positives. GONG data, NSO/AURA/NSF.
- `overlay_good_match.png`, `overlay_fragmentation.png`, `overlay_over_merge.png`, `overlay_miss.png`, `overlay_false_positive.png`: crops of holdout H-alpha frames with the annotator outline (green) and the predicted outline (orange); the filament the panel is about is drawn thicker. Examples are picked by a fixed rule and seed, not by hand. GONG data, NSO/AURA/NSF.

In [ ]:
print(f"attribution for every figure: {ATTRIBUTION}")
print("This notebook read no test image, wrote no submission and did not submit.")
if PILOT:
    print("PILOT: a plumbing check on a few files. The line below is not a result.")
summary = {row["metric"]: row["value"] for row in rows}
print(
    f"FINAL pq={summary['pq']:.6f} sq={summary['sq']:.6f} rq={summary['rq']:.6f} "
    f"mean_dice={summary['mean_dice']:.6f} one_to_many={summary['one_to_many']} "
    f"many_to_one={summary['many_to_one']}"
)